# 04 - Blocking / Candidate Generation Experiments

Goal: find blocking strategies that get **candidate recall** (fraction of true
S1->S2/S3 links that appear in the candidate set) as high as possible while
keeping **candidates/S1** small, since the final matching model has to score
every candidate and the challenge explicitly rewards a smaller candidate set.

Method: rather than indexing the full 5M+ row S2/S3 corpus for every
iteration (slow, and this machine has limited RAM), we build a **sample
corpus**:

- Sample 25,000 S1 query entities from the training ground truth (natural
  ~94% matched / ~6% singleton mix preserved).
- Collect the *full* set of their true S2/S3 matches and force-include those
  records, so recall is never lowered by an artifact of sampling.
- Add a random ~25% background sample of the rest of S2/S3, to get a
  realistic (if scaled-down) collision/candidate-volume estimate.

Because recall only depends on whether the true match shares a block key with
the query (not on how much *other* unrelated data is around), the recall
numbers below hold at full scale. Candidate-volume numbers are measured on
the ~25% sample and also reported extrapolated to the full corpus
(multiply by ~3.9x).


In [ ]:
import sys
import json
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..")
sys.path.append(str(PROJECT_ROOT / "src"))

import blocking

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DIR = PROJECT_ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


## 1. Build the sample corpus

25,000 S1 queries + guaranteed true matches + ~25% random background of
S2/S3. This takes ~90s the first time; re-run only if you want a fresh
random draw.


In [ ]:
import random

RNG_SEED = 42
N_QUERY_S1 = 25_000
BACKGROUND_KEEP_PROB = 0.25
CHUNKSIZE = 300_000

random.seed(RNG_SEED)
t0 = time.time()

gt = pd.read_csv(RAW_DIR / "train_ground_truth.tsv", sep="\t", dtype=str, keep_default_na=False)
print(f"GT rows: {len(gt):,}")

sample_gt = gt.sample(n=N_QUERY_S1, random_state=RNG_SEED).reset_index(drop=True)
sampled_s1_ids = set(sample_gt["source1_entity_id"])

ground_truth = {}
required_s2_ids, required_s3_ids = set(), set()

for row in sample_gt.itertuples(index=False):
    matches = [m.strip() for m in row.matched_entity_ids.split(",") if m.strip()] if row.matched_entity_ids else []
    ground_truth[row.source1_entity_id] = matches
    for m in matches:
        (required_s2_ids if m.startswith("S2-") else required_s3_ids).add(m)

total_links = sum(len(v) for v in ground_truth.values())
print(f"Sampled S1 queries: {len(sampled_s1_ids):,}  total true links: {total_links:,}")


In [ ]:
def stream_sample(path, required_ids, keep_prob, chunksize=CHUNKSIZE):
    rng = random.Random(RNG_SEED)
    kept = []
    for chunk in pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, chunksize=chunksize):
        is_required = chunk["entity_id"].isin(required_ids)
        mask = chunk["entity_id"].map(lambda _: rng.random() < keep_prob)
        kept.append(chunk[is_required | mask])
    return pd.concat(kept, ignore_index=True).drop_duplicates(subset="entity_id").reset_index(drop=True)


s2_sample = stream_sample(RAW_DIR / "train_source2.tsv", required_s2_ids, BACKGROUND_KEEP_PROB)
s3_sample = stream_sample(RAW_DIR / "train_source3.tsv", required_s3_ids, BACKGROUND_KEEP_PROB)
s1_sample = stream_sample(RAW_DIR / "train_source1.tsv", sampled_s1_ids, 0.0)

print(f"S1: {len(s1_sample):,}  S2: {len(s2_sample):,} (of 5,034,616)  S3: {len(s3_sample):,} (of 5,285,603)")
print(f"Elapsed: {time.time()-t0:.1f}s")

s1_sample.to_parquet(PROC_DIR / "blocking_sample_s1.parquet")
s2_sample.to_parquet(PROC_DIR / "blocking_sample_s2.parquet")
s3_sample.to_parquet(PROC_DIR / "blocking_sample_s3.parquet")
with open(PROC_DIR / "blocking_sample_ground_truth.json", "w") as f:
    json.dump(ground_truth, f)


## 2. Prepare records and build per-strategy indices

`blocking.prepare_record` normalizes name/address and computes every block
key a strategy might need (exact name, first token, all significant tokens,
token bigrams, address digit runs, country combos, ...). `blocking.STRATEGIES`
registers each one, with an optional block-purging fraction that drops any
key whose block would exceed that fraction of the corpus (a key that common
is not discriminating - it's pure candidate-volume noise).


In [ ]:
s1_records = blocking._rows_to_records(s1_sample)
s2_records = blocking._rows_to_records(s2_sample)
s3_records = blocking._rows_to_records(s3_sample)

s2_indices = {
    name: blocking.build_index(s2_records, key_field, multi, max_block_frac)
    for name, (key_field, multi, max_block_frac) in blocking.STRATEGIES.items()
}
s3_indices = {
    name: blocking.build_index(s3_records, key_field, multi, max_block_frac)
    for name, (key_field, multi, max_block_frac) in blocking.STRATEGIES.items()
}

total_links = sum(len(v) for v in ground_truth.values())
n_queries = len(s1_records)


## 3. Evaluate: candidate recall + avg candidates/S1

**Candidate recall** = fraction of true (S1, S2/S3) links whose target id
appears in the candidate set generated for that S1 (pair-level / micro
average, matching how the challenge scores blocking quality).


In [ ]:
def evaluate(strategy_names, label):
    found_links = 0
    total_candidates = total_s2 = total_s3 = 0

    for rec in s1_records:
        true_matches = ground_truth.get(rec["entity_id"], [])
        s2_cands = blocking.candidates_for_record(rec, s2_indices, strategy_names)
        s3_cands = blocking.candidates_for_record(rec, s3_indices, strategy_names)

        for m in true_matches:
            if m.startswith("S2-") and m in s2_cands:
                found_links += 1
            elif m.startswith("S3-") and m in s3_cands:
                found_links += 1

        total_s2 += len(s2_cands)
        total_s3 += len(s3_cands)
        total_candidates += len(s2_cands) + len(s3_cands)

    recall = found_links / total_links
    avg_candidates = total_candidates / n_queries
    print(f"{label:50s} recall={recall*100:6.2f}%  avg_cand/S1={avg_candidates:9.2f}  (S2={total_s2/n_queries:7.2f} S3={total_s3/n_queries:7.2f})")
    return {"strategy": label, "recall": recall, "avg_candidates_per_s1": avg_candidates}


results = [evaluate([name], name) for name in blocking.STRATEGIES]


In [ ]:
combos = {
    "union: name_token_pairs + address_digits": ("name_token_pairs", "address_digits"),
    "union: DEFAULT_UNION_STRATEGIES (production)": blocking.DEFAULT_UNION_STRATEGIES,
    "union: ALL strategies (recall ceiling)": tuple(blocking.STRATEGIES.keys()),
}
for label, combo in combos.items():
    results.append(evaluate(combo, label))


## 4. Results summary (already measured, see `data/processed/blocking_experiment_results*.json`)

### Round 1 - naive single/multi-key blocking (union OR semantics, no purging)

| Strategy | Recall | Candidates/S1 (sample) | Full-scale est. |
|---|---|---|---|
| exact_name | 22.0% | 3.1 | 12 |
| name_first_token | 77.1% | 1,773.6 | 6,900 |
| name_tokens (all tokens, OR) | 85.0% | 16,104.5 | 62,800 |
| address_first_token | 61.8% | 3,187.7 | 12,400 |
| address_digits | 57.3% | 1,294.8 | 5,000 |
| country + name_first_token | 77.1% | 1,456.6 | 5,700 |
| country + address_first_token | 61.8% | 2,854.5 | 11,100 |
| name_first_token AND address_digit | 44.7% | 2.5 | 10 |
| **UNION: name_tokens + address_digits + country_addr_first** | **97.1%** | **20,205.6** | **78,800** |

Round 1's best union hit 97% recall, but at ~78,800 candidates/S1 at full
scale (~175 **billion** S1xcandidate pairs total) - unusable. The cause:
single-token / first-token blocking keys collide on common words and common
city names, producing enormous blocks.

### Round 2 - block purging + bigram (2-token) blocking

Fix: (a) **block purging** - drop any blocking key whose block exceeds
0.05% of the corpus (common words stop being usable keys), and (b) **bigram
blocking** - require two shared significant name tokens instead of one, far
more selective while still surviving word reordering / one noisy token.

| Strategy | Recall | Candidates/S1 (sample) | Full-scale est. |
|---|---|---|---|
| name_first_token_purged | 47.1% | 107.3 | 418 |
| name_tokens_purged | 56.5% | 281.1 | 1,096 |
| name_token_pairs (bigram) | 71.1% | 374.6 | 1,461 |
| address_first_token_purged | 47.4% | 181.2 | 706 |
| country_name_first_purged | 48.1% | 121.5 | 474 |
| country_address_first_purged | 48.2% | 182.5 | 712 |
| union: name_token_pairs + address_digits | 86.7% | 1,667.7 | 6,504 |
| union: pairs + digits + country_addr_purged | 92.5% | 1,847.5 | 7,205 |
| **union: pairs + digits + country_addr_purged + name_first_purged** | **94.4%** | **1,944.8** | **7,589** |

**~10x fewer candidates for only 2.7pp less recall** than round 1's best
union. This combo is `blocking.DEFAULT_UNION_STRATEGIES` and is what
`generate_candidates_s2` / `generate_candidates_s3` use by default.

### Recommendation

Ship the round-2 default union (`name_token_pairs`, `address_digits`,
`country_address_first_purged`, `name_first_token_purged`) as the blocking
stage feeding the matching model: **94.4% candidate recall at ~7,600
candidates/S1** (full-scale estimate). The remaining ~5.6% of true links
that slip through are cases where the name is too garbled to share even one
significant token pair *and* the address digit/first-token/country signals
also disagree - likely needs a phonetic (soundex/metaphone) or fuzzy edit-
distance top-up pass if we want to push recall further, at the cost of more
candidates. Worth a follow-up experiment once the matching model's precision
budget under F0.5 is known - it can absorb a modest recall/precision
trade-off, so an F0.5-informed choice makes sense here.
